[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ELTE-DSED/Intro-Data-Security/blob/main/modules/module_07_defenses/Lab_7a_Differential_Privacy_and_DP_SGD.ipynb)

# **Lab 7a: Differential Privacy & DP-SGD with Opacus**

**Master's Course:** Introduction to Data Security Practicum  
**Institution:** Eötvös Loránd University (ELTE), Faculty of Informatics (IK)  
**Module 7:** Provable Privacy Defenses


## **Learning Objectives**

By the end of this lab, you will be able to:

- **Understand Differential Privacy Foundations:** Formulate the mathematical definition of $(\epsilon, \delta)$-Differential Privacy, the Privacy Loss Random Variable, and the Gaussian Mechanism.
- **Implement First-Principles DP-SGD:** Build an end-to-end DP-SGD optimization step in pure PyTorch incorporating per-sample gradient clipping, Gaussian noise calibration, and gradient accumulation.
- **Master Production DP-SGD with Opacus:** Instrument standard PyTorch models with Meta's Opacus `PrivacyEngine`, leveraging vectorized backward hooks and the Rényi Differential Privacy (RDP) accountant.
- **Analyze Privacy-Utility Trade-offs:** Systematically sweep noise multipliers $\sigma \in [0.1, 2.0]$ to map empirical test accuracy against formal privacy spend $\epsilon$.
- **Audit Privacy Preservation via MIA:** Empirically quantify resistance to Membership Inference Attacks (MIA), validating that provable DP bounds collapse attacker advantage toward random guessing ($50\%$).


## **Table of Contents**

1. [Threat Model & Foundations of $(\epsilon, \delta)$-Differential Privacy](#threat-model)
2. [Setup & Data Preparation (MNIST Benchmark)](#setup)
3. [Model Architecture & Non-Private Baseline](#baseline)
4. [First-Principles DP-SGD (Per-Sample Clipping & Gaussian Noise)](#dpsgd)
5. [Industry-Standard DP-SGD with Opacus](#opacus)
6. [Sensitivity Analysis: Noise vs. Utility Trade-off](#sensitivity)
7. [Empirical Privacy Auditing via Membership Inference](#mia)
8. [Hands-on Exploratory Exercises](#exercises)
9. [Conclusion & Next Steps](#conclusion)


## **1. Threat Model & Foundations of $(\epsilon, \delta)$-Differential Privacy** <a name="threat-model"></a>

Classical heuristic defenses (e.g., $L_2$ weight decay, dropout, output rounding) reduce empirical overfitting but offer **no mathematical guarantees** against an adversary with arbitrary auxiliary information or infinite computational resources.

### **Mathematical Formulation**

A randomized training algorithm $\mathcal{M}$ satisfies $(\epsilon, \delta)$-**Differential Privacy** if for any two neighboring datasets $\mathcal{D}, \mathcal{D}'$ differing by at most one individual sample ($|\mathcal{D} \Delta \mathcal{D}'| = 1$), and for any set of model output parameters $\mathcal{S} \subseteq \text{Range}(\mathcal{M})$:

$$\mathbb{P}[\mathcal{M}(\mathcal{D}) \in \mathcal{S}] \le e^{\epsilon} \mathbb{P}[\mathcal{M}(\mathcal{D}') \in \mathcal{S}] + \delta$$

- **$\epsilon$ (Privacy Budget):** Upper bounds the multiplicative privacy leakage per query. Smaller $\epsilon$ means stronger privacy protection ($"\epsilon=0"$ is pure independent random output).
- **$\delta$ (Failure Probability):** The probability that the $\epsilon$-bound fails, typically set strictly smaller than $\frac{1}{|\mathcal{D}|}$ (e.g., $10^{-5}$).

### **DP-SGD: Differentially Private Stochastic Gradient Descent**

Introduced by [Abadi et al. (CCS 2016)](https://doi.org/10.1145/2976749.2978318), DP-SGD modifies standard mini-batch SGD through three vital operations:

1. **Per-Sample Gradient Computation:** Compute individual gradient $\mathbf{g}_i = \nabla_\theta \mathcal{L}(\mathbf{x}_i, y_i; \theta)$ for each sample $i$ in mini-batch $\mathcal{B}$.
2. **Per-Sample Gradient Clipping:** Bound individual influence by scaling gradients to a maximum $L_2$ threshold $C$:
   $$\bar{\mathbf{g}}_i = \mathbf{g}_i \cdot \min\left(1, \frac{C}{\|\mathbf{g}_i\|_2}\right)$$
3. **Noise Perturbation & Gradient Descent:** Add calibrated spherical Gaussian noise proportional to sensitivity $C$:
   $$\tilde{\mathbf{g}} = \frac{1}{|\mathcal{B}|} \left( \sum_{i \in \mathcal{B}} \bar{\mathbf{g}}_i + \mathcal{N}(0, \sigma^2 C^2 \mathbf{I}) \right)$$
   $$\theta_{t+1} = \theta_t - \eta \tilde{\mathbf{g}}$$


## **2. Setup & Data Preparation (MNIST Benchmark)** <a name="setup"></a>

We configure dependencies, install Meta's **Opacus** library, and load a curated subset of the **MNIST** handwritten digits dataset ($1,000$ training samples, $500$ evaluation samples).


In [ ]:
# Install required dependencies for Google Colab
%pip install -q opacus scikit-learn matplotlib torchvision


In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
import torchvision.transforms as transforms
from torchvision.datasets import MNIST

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    plt.style.use('seaborn-v0_8-muted')
except:
    plt.style.use('seaborn-muted')
from sklearn.metrics import roc_auc_score
from dataclasses import dataclass

np.random.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True) 

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = MNIST(root='./data', train=False, download=True, transform=transform)

train_indices = np.random.choice(len(train_dataset), 1000, replace=False)
test_indices = np.random.choice(len(test_dataset), 500, replace=False)

train_data = Subset(train_dataset, train_indices)
test_data = Subset(test_dataset, test_indices)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

print(f"\nTrain: {len(train_data)}, Test: {len(test_data)}")

## **3. Model Architecture & Non-Private Baseline** <a name="baseline"></a>

We define a 2-layer perceptron (`MLP`) suitable for both first-principles autograd and Opacus module hooks. We first train a non-private baseline using standard SGD.


In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, 512)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 10)
    
    def forward(self, x):
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

In [ ]:
def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            pred = output.argmax(dim=1)
            correct += (pred == target).sum().item()
            total += target.size(0)
    return 100.0 * correct / total

In [ ]:
def train_baseline(model, loader, epochs=5):
    model.to(device)
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=0)
    criterion = nn.CrossEntropyLoss()
    losses = []
    for epoch in range(epochs):
        epoch_loss = 0
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
        losses.append(epoch_loss / len(loader))
        print(f"Epoch {epoch+1}/{epochs}: Loss={losses[-1]:.4f}")
    return losses

In [ ]:
baseline_model = MLP()
baseline_losses = train_baseline(baseline_model, train_loader, epochs=30)

baseline_train_acc = evaluate(baseline_model, train_loader)
baseline_test_acc = evaluate(baseline_model, test_loader)

print(f"\nBaseline Accuracy:")
print(f"  Train: {baseline_train_acc:.2f}%")
print(f"  Test:  {baseline_test_acc:.2f}%")

## **4. First-Principles DP-SGD (Per-Sample Clipping & Gaussian Noise)** <a name="dpsgd"></a>

To demystify how privacy guarantees are injected at the tensor level, we implement `dp_sgd_step` manually in PyTorch. For each sample in the mini-batch, we:
1. Compute the analytical gradient with `torch.autograd.grad`.
2. Clip the per-sample gradient norm to $C = 1.0$.
3. Sum the clipped gradients and add Gaussian noise with scale $\sigma C$.


In [ ]:
@dataclass
class DPConfig:
    max_grad_norm: float = 1.0
    noise_multiplier: float = 1.0  # σ
    epochs: int = 5
    batch_size: int = 64
    lr: float = 0.01

In [ ]:
def dp_sgd_step(model, data, target, config: DPConfig):
    """Single DP-SGD step with per-sample gradient clipping + noise."""
    model.train()
    criterion = nn.CrossEntropyLoss(reduction='none')
    
    data, target = data.to(device), target.to(device)
    batch_size = data.size(0)
    
    # Compute per-sample gradients
    batch_loss = 0.0
    per_sample_grads = []
    for i in range(batch_size):
        model.zero_grad()
        output = model(data[i:i+1])
        loss = criterion(output, target[i:i+1]).mean()
        batch_loss += loss.item()
        loss.backward()
        grads = [p.grad.detach().clone() for p in model.parameters()]
        per_sample_grads.append(grads)
    
    # Clip per-sample gradients
    clipped_grads = []
    for grads in per_sample_grads:
        total_norm = torch.sqrt(sum((g**2).sum() for g in grads))
        clip_factor = min(1.0, config.max_grad_norm / (total_norm + 1e-6))
        clipped = [g * clip_factor for g in grads]
        clipped_grads.append(clipped)
    
    # Aggregate clipped gradients
    agg_grads = []
    for param_i in range(len(clipped_grads[0])):
        stacked = torch.stack([g[param_i] for g in clipped_grads], dim=0)
        agg = stacked.mean(dim=0)
        agg_grads.append(agg)
    
    # Add noise
    noisy_grads = []
    for g in agg_grads:
        noise = torch.randn_like(g) * (config.noise_multiplier * config.max_grad_norm / batch_size)
        noisy_grads.append(g + noise)
    
    # Apply gradients
    for param, grad in zip(model.parameters(), noisy_grads):
        param.grad = grad
    return batch_loss / batch_size

In [ ]:
def train_dp_sgd(model, loader, config: DPConfig):
    optimizer = optim.SGD(model.parameters(), lr=config.lr, momentum=0.9)
    losses = []
    for epoch in range(config.epochs):
        epoch_loss = 0
        for data, target in loader:
            optimizer.zero_grad()
            step_loss = dp_sgd_step(model, data, target, config)
            optimizer.step()
            # Compute loss for logging
            epoch_loss += step_loss
        losses.append(epoch_loss / len(loader))
        print(f"Epoch {epoch+1}/{config.epochs}: Loss={losses[-1]:.4f}")
    return losses

In [ ]:
dp_config = DPConfig(max_grad_norm=1.0, noise_multiplier=0.8, epochs=30, lr=0.02)
dp_model = MLP().to(device)
dp_losses = train_dp_sgd(dp_model, train_loader, dp_config)

dp_train_acc = evaluate(dp_model, train_loader)
dp_test_acc = evaluate(dp_model, test_loader)

print(f"\nDP-SGD Accuracy:")
print(f"  Train: {dp_train_acc:.2f}%")
print(f"  Test:  {dp_test_acc:.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(baseline_losses, label='Baseline', color='#e74c3c', alpha=0.5)
axes[0].plot(dp_losses, label='DP-SGD', color='#3498db', linewidth=2)
axes[0].set_title('Training Loss Comparison')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].bar(['Baseline', 'DP-SGD'], [baseline_test_acc, dp_test_acc], color=['#e74c3c', '#3498db'])
axes[1].set_title('Utility Trade-off (Test Accuracy)')
axes[1].set_ylabel('Accuracy (%)')
axes[1].set_ylim(0, 100)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## **5. Production DP-SGD with Opacus PrivacyEngine** <a name="opacus"></a>

While manual DP-SGD is pedagogically illustrative, computing per-sample gradients via a sequential loop is computationally prohibitive.

[Opacus](https://opacus.ai/) is a high-speed library developed by Meta that optimizes DP-SGD using vectorized backward hooks (via PyTorch's `functorch`). It also integrates the **Rényi Differential Privacy (RDP)** accountant to calculate exact, composable $(\epsilon, \delta)$ values at every training epoch.


In [ ]:
from opacus import PrivacyEngine
from opacus.utils.batch_memory_manager import BatchMemoryManager

# 1. Initialize model, optimizer, and data loader
opacus_model = MLP().to(device)
opacus_optimizer = optim.SGD(opacus_model.parameters(), lr=0.02, momentum=0.9)
opacus_train_loader = DataLoader(train_data, batch_size=64, shuffle=True)

# 2. Attach Privacy Engine
privacy_engine = PrivacyEngine()
model, optimizer, train_loader = privacy_engine.make_private(
    module=opacus_model,
    optimizer=opacus_optimizer,
    data_loader=opacus_train_loader,
    noise_multiplier=0.8,
    max_grad_norm=1.0,
)

print(f"Privatized model using σ={0.8} and C={1.0}")

In [ ]:
def train_opacus(model, optimizer, loader, epochs=20):
    model.train()
    criterion = nn.CrossEntropyLoss()
    losses = []
    epsilons = []
    
    # Using BatchMemoryManager is best practice for large batches in DP
    with BatchMemoryManager(
        data_loader=loader, max_physical_batch_size=64, optimizer=optimizer
    ) as memory_safe_loader:
        for epoch in range(epochs):
            epoch_loss = 0
            for data, target in memory_safe_loader:
                data, target = data.to(device), target.to(device)
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, target)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            epsilon = privacy_engine.get_epsilon(delta=1e-5)
            losses.append(epoch_loss / len(loader))
            epsilons.append(epsilon)
            if (epoch + 1) % 5 == 0:
                print(f"Epoch {epoch+1}/{epochs}: Loss={losses[-1]:.4f}, ε={epsilon:.2f}")
    return losses, epsilons

In [ ]:
opacus_losses, epsilons = train_opacus(model, optimizer, train_loader, epochs=20)
opacus_test_acc = evaluate(model, test_loader)

plt.figure(figsize=(8, 4))
plt.plot(epsilons, marker='o', color='#2ecc71', linewidth=2)
plt.title('Epsilon (Privacy Budget) Growth Over Epochs')
plt.xlabel('Epoch')
plt.ylabel('ε (at δ=1e-5)')
plt.grid(alpha=0.3)
plt.show()

print(f"\nOpacus DP-SGD Test Accuracy: {opacus_test_acc:.2f}%")

## **6. Sensitivity Analysis: Noise vs. Utility Trade-off** <a name="sensitivity"></a>

A fundamental theorem of Differential Privacy is the **Privacy-Utility Tradeoff**: increasing noise multiplier $\sigma$ yields tighter privacy bounds (lower $\epsilon$) at the cost of empirical test accuracy.

Below, we sweep $\sigma \in \{0.1, 0.5, 1.0, 2.0\}$ and plot the resulting Pareto curve.


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", message=".*Secure RNG turned off.*")
warnings.filterwarnings("ignore", message=".*Using a non-full backward hook.*")

noises = [0.1, 0.5, 1.0, 2.0]
results = []

print("Running Sensitivity Analysis...")
for sigma in noises:
    # Re-init model and engine
    m = MLP().to(device)
    opt = optim.SGD(m.parameters(), lr=0.02, momentum=0.9)
    pe = PrivacyEngine()
    m_priv, opt_priv, dl_priv = pe.make_private(
        module=m, optimizer=opt, data_loader=opacus_train_loader, 
        noise_multiplier=sigma, max_grad_norm=1.0
    )
    
    # Train for fewer epochs for speed
    _ = train_opacus(m_priv, opt_priv, dl_priv, epochs=10)
    acc = evaluate(m_priv, test_loader)
    eps = pe.get_epsilon(delta=1e-5)
    results.append({'noise': sigma, 'accuracy': acc, 'epsilon': eps})
    print(f"σ={sigma}: Acc={acc:.2f}%, ε={eps:.2f}")

res_df = pd.DataFrame(results)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()

ax1.plot(res_df['noise'], res_df['accuracy'], marker='o', color='#3498db', label='Accuracy')
ax2.plot(res_df['noise'], res_df['epsilon'], marker='s', color='#e74c3c', label='Epsilon')

ax1.set_xlabel('Noise Multiplier (σ)')
ax1.set_ylabel('Accuracy (%)', color='#3498db')
ax2.set_ylabel('Epsilon (ε)', color='#e74c3c')
plt.title('Sensitivity Analysis: The Privacy-Utility Trade-off')
plt.grid(alpha=0.3)
plt.show()

## **7. Empirical Privacy Auditing via Membership Inference** <a name="mia"></a>

Can an adversary infer whether an individual record was part of the training dataset?

We audit both the **Non-Private Baseline Model** and the **DP-SGD Model** using the prediction loss distribution. An effective privacy defense collapses the Membership Inference ROC-AUC from elevated levels toward $0.50$ (random coin flip).


In [ ]:
def get_mia_scores(model, loader):
    model.eval()
    scores = []
    criterion = nn.CrossEntropyLoss(reduction='none')
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            loss = criterion(output, target)
            # Use negative loss as score (higher = more likely to be in training set)
            scores.extend((-loss).cpu().numpy())
    return np.array(scores)

In [ ]:
# Baseline model MIA
baseline_train_scores = get_mia_scores(baseline_model, train_loader)
baseline_test_scores = get_mia_scores(baseline_model, test_loader)
labels_base = np.concatenate([np.ones(len(baseline_train_scores)), np.zeros(len(baseline_test_scores))])
scores_base = np.concatenate([baseline_train_scores, baseline_test_scores])
auc_base = roc_auc_score(labels_base, scores_base)

# Opacus DP model MIA
opacus_train_scores = get_mia_scores(opacus_model, train_loader)
opacus_test_scores = get_mia_scores(opacus_model, test_loader)
labels_opacus = np.concatenate([np.ones(len(opacus_train_scores)), np.zeros(len(opacus_test_scores))])
scores_opacus = np.concatenate([opacus_train_scores, opacus_test_scores])
auc_opacus = roc_auc_score(labels_opacus, scores_opacus)

# Manual DP model MIA
dp_train_scores = get_mia_scores(dp_model, train_loader)
dp_test_scores = get_mia_scores(dp_model, test_loader)
labels_dp = np.concatenate([np.ones(len(dp_train_scores)), np.zeros(len(dp_test_scores))])
scores_dp = np.concatenate([dp_train_scores, dp_test_scores])
auc_dp = roc_auc_score(labels_dp, scores_dp)

print("\nMembership Inference Audit:")
print(f"  Baseline AUC: {auc_base:.4f}")
print(f"  DP-SGD AUC:   {auc_dp:.4f}")

## **8. Hands-on Exploratory Exercises** <a name="exercises"></a>

### **Exercise 1: Clipping Threshold Sensitivity ($C$)**
Fix the noise multiplier $\sigma = 1.0$ and evaluate model accuracy across three clipping thresholds: $C \in \{0.1, 1.0, 10.0\}$.
- What happens when $C$ is too small (e.g., $0.1$)?
- What happens when $C$ is too large (e.g., $10.0$)?
- Why is adaptive gradient clipping useful in production deployments?

### **Exercise 2: Batch Size & Privacy Amplification by Subsampling**
In DP-SGD, drawing small random batches from a large dataset amplifies privacy by a factor of $q = \frac{B}{N}$ (the subsampling ratio).
Train Opacus models with batch sizes $B \in \{16, 64, 256\}$ for the same number of total optimization steps. How does batch size influence the final $(\epsilon, \delta)$ bound?

### **Exercise 3: Disparate Impact of DP-SGD on Minority Subclasses**
Audit the class-wise accuracy of the baseline model vs. the DP-SGD model across all 10 digit categories. Does gradient clipping disproportionately penalize difficult-to-classify categories (e.g., digit `8` vs. digit `1`)? How can fairness-aware DP mitigate this discrepancy?

### **Exercise 4: Comparing Privacy Accountants (RDP vs. GDP vs. Moments Accountant)**
Research and contrast the Rényi Differential Privacy (RDP) accountant used in Opacus with Gaussian Differential Privacy (GDP) and the original Moments Accountant (Abadi et al., 2016). Which accountant produces the tightest $\epsilon$ bound for 100+ training epochs?


## **9. Conclusion & Next Steps** <a name="conclusion"></a>

### **Core Takeaways**
1. **Mathematical Guarantees Trump Heuristics:** Unlike weight decay or output rounding, $(\epsilon, \delta)$-Differential Privacy provides worst-case protection against adversaries with arbitrary auxiliary information.
2. **Three Pillars of DP-SGD:** DP-SGD functions by computing per-sample gradients, clipping each gradient's $L_2$ norm to bound individual sensitivity, and injecting calibrated Gaussian noise before the parameter update.
3. **Opacus Provides Production Scaling:** By implementing vectorized backward hooks and the Rényi Differential Privacy (RDP) accountant, Opacus enables DP training with minimal runtime overhead compared to standard PyTorch SGD.
4. **Empirical Verification via MIA Auditing:** Differential Privacy provably collapses Membership Inference Attack advantage: an auditor observing model outputs cannot distinguish training members from unseen test records with high statistical confidence ($AUC \approx 0.50$).

### **MITRE ATLAS Mapping**
- **Mitigation:** [AML.M1006: Differential Privacy Training](https://atlas.mitre.org/mitigations/AML.M1006)
- **Tactic Countered:** [AML.TA0005: Exfiltration](https://atlas.mitre.org/tactics/AML.TA0005)
- **Technique Countered:** [AML.T0024: Infer Training Data Membership](https://atlas.mitre.org/techniques/AML.T0024)

### **Key References**
- Abadi, M., Chu, A., Goodfellow, I., McMahan, H. B., Mironov, I., Talwar, K., & Zhang, L. (2016). *Deep Learning with Differential Privacy*. ACM CCS. [DOI:10.1145/2976749.2978318](https://doi.org/10.1145/2976749.2978318).
- Mironov, I. (2017). *Rényi Differential Privacy*. IEEE CSF. [arXiv:1702.07476](https://arxiv.org/abs/1702.07476).
- Yousefpour, A., et al. (2021). *Opacus: User-Friendly Differential Privacy Library in PyTorch*. NeurIPS Workshop. [arXiv:2109.12298](https://arxiv.org/abs/2109.12298).
